# PHYS 405 Computerized Data Analysis · Week 5 · Poisson and Bernoulli processes
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs5/Lab5.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Run the code, then **Compare** with your prediction.
3. Run the two checks. Each check tests the answer a different way.
4. Do the **Your turn** task in the empty cell.

Today's physics: a trigger, a rare-event search, a million nuclei, a Geiger counter's background, arrival gaps, a detector efficiency, a count in a signal region and a counter's dead time. All simulated numbers use `np.random.default_rng(seed)` with the seed given in each problem, so your printed numbers match the slides.
The lab uses new data: the lecture's numbers do not carry over.

**Your Drive folder.** The next cell connects Google Drive (Colab asks for permission once) and makes the folder
`MyDrive/PHY405_Labs/Week05` if it does not exist. Every file this notebook writes goes there.

In [ ]:
import os
try:
    from google.colab import drive                      # running in Colab
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/PHY405_Labs/Week05"
except ImportError:                                     # running elsewhere: a local folder
    LAB_DIR = os.path.join(os.getcwd(), "PHY405_Labs", "Week05")
os.makedirs(LAB_DIR, exist_ok=True)
os.chdir(LAB_DIR)                                       # files written below land here
print("working folder:", os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%config InlineBackend.figure_format = 'retina'
print("numpy", np.__version__)

## Problem 1 · Waiting for the first trigger
A trigger fires on each event independently with probability 0.05 (illustrative). What is the probability that it fires at least once in the first 10 events? What is the mean wait?

*Givens:* $p = 0.05$ per event; independent events; seed 1.

**Predict.** Is it $10 \times 0.05 = 0.50$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
p = 0.05
print(round(1 - (1 - p)**10, 3), 10 * p)          # formula; trap
rng = np.random.default_rng(1)
k = rng.geometric(p, 100_000)                      # events until the first trigger
print(round((k <= 10).mean(), 3), round(k.mean(), 2))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. With 30 events, what would the sum give?

In [ ]:
# check A (simulation): 100 000 waits (seed 1) agree with 1 - 0.95^10 to 0.01
assert abs((k <= 10).mean() - (1 - 0.95**10)) < 0.01
print("check A passed")

In [ ]:
# check B (inequality): more than one event's chance, less than the sum of ten
assert 0.05 < 1 - 0.95**10 < 0.50
print("check B passed")

**Your turn.** With the same trigger, what is the probability of at least one trigger within 20 events? After how many events has half of all waits ended (the median)?

*Hint:* $1 - 0.95^n$; find the smallest $n$ with $1 - 0.95^n \ge 0.5$.

In [ ]:
# TODO: write your answer here

## Problem 2 · No memory
A rare-event search: each one-hour run has an event with probability 0.02 (illustrative). The first 30 runs were empty. What is the probability of an event in the next 10 runs?

*Givens:* $p = 0.02$ per run; independent runs; seed 2.

**Predict.** After 30 empty runs, is an event in the next 10 more likely than for a fresh start? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
p = 0.02
print(round(1 - (1 - p)**10, 3))                    # a fresh start
rng = np.random.default_rng(2)
k = rng.geometric(p, 200_000)
late = k[k > 30]                                     # after 30 empty runs
print(len(late), round((late <= 40).mean(), 3), round((late - 30).mean(), 1))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. What did the 30 empty runs change in the answer?

In [ ]:
# check A (simulation): the sequences that passed 30 empty runs give the fresh-start probability
assert abs((late <= 40).mean() - (1 - 0.98**10)) < 0.01
print("check A passed")

In [ ]:
# check B (memoryless): the mean further wait is 1/p = 50 runs, as from the start
assert abs((late - 30).mean() - 50) < 1.5
print("check B passed")

**Your turn.** After the 30 empty runs, how many more runs should the experiment expect to wait, on average, for its first event?

*Hint:* memoryless; the mean wait of a geometric law is $1/p$.

In [ ]:
# TODO: write your answer here

## Problem 3 · From binomial to Poisson
A sample has $10^6$ nuclei; each decays within 10 s with probability $3 \times 10^{-6}$ (illustrative). Compare the binomial with Poisson(3). What are $P(0)$ and $P(N \ge 5)$?

*Givens:* binomial $n = 10^6$, $p = 3 \times 10^{-6}$; $\lambda = np = 3$.

**Predict.** Can a 10 s window have no decay at all, if on average 3 decay? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
k = np.arange(8)
b = stats.binom.pmf(k, 10**6, 3e-6)                  # a million nuclei
q = stats.poisson.pmf(k, 3)
print(np.round(b[:4], 4), np.round(q[:4], 4))
print(f"{np.abs(b - q).max():.1e}", round(1 - q[:5].sum(), 4))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. How many empty windows do you expect in 100?

In [ ]:
# check A (independent method): the exact binomial and Poisson agree to 1e-6
assert np.abs(b - q).max() < 1e-6
print("check A passed")

In [ ]:
# check B (normalisation): the Poisson probabilities add to 1
assert abs(stats.poisson.pmf(np.arange(60), 3).sum() - 1) < 1e-12
print("check B passed")

**Your turn.** A weaker sample gives $\lambda = 0.5$ decays per 10 s. What are $P(0)$, $P(1)$ and $P(N \ge 2)$?

*Hint:* $P(k) = 0.5^k e^{-0.5}/k!$; $P(N \ge 2) = 1 - P(0) - P(1)$.

In [ ]:
# TODO: write your answer here

## Problem 4 · Is it Poisson?
A Geiger counter records background in 100 intervals of 10 s (simulated). Compute the mean, the variance and their ratio. How well is the mean known?

*Givens:* Poisson ⇒ variance = mean; error of a mean = $\sigma/\sqrt n$ (Week 2); seed 53.

**Predict.** Is the error of the mean $\sqrt{\text{mean}}$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(53)
c = rng.poisson(3.1, 100)                            # counts per 10 s
m, v = c.mean(), c.var(ddof=1)
print(round(m, 2), round(v, 2), round(v / m, 2))
print(round(np.sqrt(m / 100), 3), round(np.sqrt(m), 2))   # error of the mean; trap

In [ ]:
plt.figure(figsize=(7, 3.5))
kk = np.arange(11)
plt.hist(c, bins=np.arange(-0.5, 11), color="tab:blue", label="100 intervals")
plt.plot(kk, 100 * stats.poisson.pmf(kk, m), "o", color="tab:green", label="Poisson(mean) × 100")
plt.xlabel("counts per 10 s"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Which number shrinks as you take more intervals: the spread or the error of the mean?

In [ ]:
# check A (Poisson test): variance / mean is close to 1
assert abs(v / m - 1) < 0.3
print("check A passed")

In [ ]:
# check B (total count): sqrt(total)/n gives the same error of the mean
assert abs(np.sqrt(c.sum()) / 100 - np.sqrt(m / 100)) < 1e-12
print("check B passed")

**Your turn.** Turn the same data into a background rate per second with its error.

*Hint:* total counts over total time; the error of a count is $\sqrt N$.

In [ ]:
# TODO: write your answer here

## Problem 5 · Gaps of a Poisson process
Events arrive as a Poisson process at 2 per second (simulated). What fraction of the gaps between events is shorter than 0.1 s, a fifth of the mean gap?

*Givens:* gaps exponential with mean $1/r = 0.5$ s; $P(\text{gap} < t) = 1 - e^{-rt}$; seed 5.

**Predict.** Are gaps shorter than a fifth of the mean rare, about 5 %? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(5)
g = rng.exponential(0.5, 10_000)                     # gaps in s, rate 2 per s
print(round(1 - np.exp(-0.2), 3), round((g < 0.1).mean(), 3))
print(round(g.mean(), 3), round(np.median(g), 3))

In [ ]:
plt.figure(figsize=(7, 3.5))
t = np.linspace(0, 2, 201)
plt.hist(g, bins=np.arange(0, 2.0001, 0.05), density=True, color="tab:blue", label="10 000 gaps")
plt.plot(t, 2 * np.exp(-2 * t), color="tab:green", label=r"$2e^{-2t}$")
plt.axvline(0.1, color="tab:red", ls="--", label="0.1 s")
plt.xlabel("gap (s)"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Where is the tallest bar of the gap histogram?

In [ ]:
# check A (simulation): agrees with 1 - e^-0.2 to 0.01
assert abs((g < 0.1).mean() - (1 - np.exp(-0.2))) < 0.01
print("check A passed")

In [ ]:
# check B (order): the median gap (ln 2 x 0.5 s) lies below the mean
assert np.median(g) < g.mean()
print("check B passed")

**Your turn.** With the same process, what fraction of the gaps is longer than 1.5 s (three times the mean)?

*Hint:* $P(\text{gap} > t) = e^{-rt}$.

In [ ]:
# TODO: write your answer here

## Problem 6 · Efficiency keeps it Poisson
Muons arrive at 50 per minute on average (Poisson); a detector sees each with probability 0.3 (illustrative). What are the mean and the variance of the detected count per minute?

*Givens:* arrivals Poisson(50); each detected with probability 0.3, independently; seed 6.

**Predict.** Is the variance the binomial $50 \times 0.3 \times 0.7 = 10.5$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(6)
s = rng.poisson(50, 100_000)                          # muons per minute
d = rng.binomial(s, 0.3)                              # detected
print(round(d.mean(), 2), round(d.var(), 2), round(50 * 0.3 * 0.7, 1))

In [ ]:
plt.figure(figsize=(7, 3.5))
kk = np.arange(36)
plt.hist(d, bins=np.arange(-0.5, 36), color="tab:blue", label="100 000 minutes")
plt.plot(kk, 1e5 * stats.poisson.pmf(kk, 15), color="tab:green", label="Poisson(15)")
plt.plot(kk, 1e5 * stats.binom.pmf(kk, 50, 0.3), "--", color="tab:red", label="binomial(50, 0.3)")
plt.xlabel("detected muons per minute"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Which curve fits the histogram?

In [ ]:
# check A (simulation): the variance is close to 15, not 10.5
assert abs(d.var() - 15) < 0.5
print("check A passed")

In [ ]:
# check B (MGF): averaging (0.7 + 0.3 e^t)^N over Poisson(50) gives exp(15 (e^t - 1)); compare at t = 0.1
t = 0.1
lhs = np.sum(stats.poisson.pmf(np.arange(200), 50) * (0.7 + 0.3 * np.exp(t))**np.arange(200))
assert abs(lhs - np.exp(15 * (np.exp(t) - 1))) < 1e-9
print("check B passed")

**Your turn.** Replace the detector by one with efficiency 0.6. What are the mean and the variance of the detected count, and how often is a minute empty?

*Hint:* Poisson($\varepsilon\lambda$); $P(0) = e^{-\varepsilon\lambda}$.

In [ ]:
# TODO: write your answer here

## Problem 7 · A count in the signal region
The background estimate in a signal region is $\hat B = 2.5$ events (illustrative); $N_{SR} = 7$ are seen. How often does background alone give 7 or more? Estimate $S$ with its error.

*Givens:* background-only $N \sim$ Poisson(2.5); $S = N_{SR} - \hat B$, $\sigma_S = \sqrt{N_{SR}}$ for an exact $\hat B$; seed 7.

**Predict.** Is the answer $P(N = 7)$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
B, N = 2.5, 7
print(round(stats.poisson.sf(N - 1, B), 4), round(stats.poisson.pmf(N, B), 4))
rng = np.random.default_rng(7)
n = rng.poisson(B, 1_000_000)
print(round((n >= N).mean(), 4), N - B, round(np.sqrt(N), 2))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. `sf(N - 1)` means "N or more". Why minus one?

In [ ]:
# check A (simulation): 1e6 background-only experiments agree with the formula to 0.001
assert abs((n >= N).mean() - stats.poisson.sf(N - 1, B)) < 0.001
print("check A passed")

In [ ]:
# check B (inequality): "7 or more" is more likely than "exactly 7"
assert stats.poisson.sf(N - 1, B) > stats.poisson.pmf(N, B)
print("check B passed")

**Your turn.** The background was estimated from sidebands: $N_{SB} = 10$ events in a region $\tau = 4$ times wider than the signal region. What are $\hat B$ and its error, and $\sigma_S$?

*Hint:* $\hat B = N_{SB}/\tau$, $\sigma_{\hat B} = \sqrt{N_{SB}}/\tau$; $\sigma_S = \sqrt{N_{SR} + N_{SB}/\tau^2}$.

In [ ]:
# TODO: write your answer here

## Problem 8 · Dead time
Events arrive at a true rate of 200 per s; the counter is dead for 1 ms after each recorded event (non-paralysable). What rate does it read, and how do you correct it?

*Givens:* $m = r/(1 + r\tau_d)$, $r = m/(1 - m\tau_d)$; seed 8.

**Predict.** Does the counter read 200 per s, since 1 ms is short? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
r, td = 200.0, 1e-3; m = r / (1 + r * td)            # per s, s
print(round(m, 1), round(m / (1 - m * td), 1))
rng = np.random.default_rng(8)
t = np.cumsum(rng.exponential(1 / r, 25_000)); t = t[t < 100]
kept, last = 0, -1.0
for x in t:
    if x - last >= td: kept, last = kept + 1, x
print(len(t), kept, round(kept / 100, 1))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. What fraction of the time is the counter dead?

In [ ]:
# check A (simulation): the recorded rate agrees with r/(1 + r tau_d) to 2 %
assert abs(kept / 100 - m) / m < 0.02
print("check A passed")

In [ ]:
# check B (limiting case): with tau_d -> 0 the counter reads the true rate
assert abs(r / (1 + r * 1e-12) - r) < 1e-6
print("check B passed")

**Your turn.** A counter with $\tau_d = 5$ ms reads 100 per s. What is the true rate, and what fraction of events does it lose?

*Hint:* $r = m/(1 - m\tau_d)$; the dead fraction is $m\tau_d$.

In [ ]:
# TODO: write your answer here

## Before you close the tab
- Mistakes we met: adding probabilities (P1), "overdue" (P2), $P(0) = 0$ (P3), $\sqrt{\text{mean}}$ as the error of the mean (P4), "short gaps are rare" (P5), a binomial variance with a fixed $n$ (P6), $P(N = 7)$ for "7 or more" (P7), reading a dead-time counter as the true rate (P8).
- Each has a check: a complement, a simulation with the same seed, variance/mean = 1 for Poisson.
- **File → Save a copy in Drive** if you have not yet. Next week: the midterm (Week 6).